# Modelo Multimodal con Transfer Learning (ImageNet)

Este cuaderno implementa la experimentación con el modelo **MTDE-Net** utilizando **Transfer Learning** basado en backbones convolucionales preentrenados en **ImageNet** (ResNet-18 y ResNet-50).

### Enfoque Científico del Experimento:
* Las imágenes térmicas son canal único (escala de grises), por lo que se expanden a 3 canales en la primera capa convolucional de manera implícita para conservar el 100% de los pesos preentrenados de ImageNet.
* Evaluamos tres configuraciones de entrenamiento:
  1. **ResNet-18 con Fine-tuning completo** (backbone preentrenado sintonizado con learning rate bajo).
  2. **ResNet-18 con Backbone Congelado** (usado puramente como extractor de características estáticas).
  3. **ResNet-50 con Fine-tuning completo** (backbone de mayor capacidad no lineal).

Al final, se comparan las métricas obtenidas con las del MTDE-Net original sin preentrenar.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net_tl import MTDE_Net_TL
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.utils import SqrtScaledMSELoss, eval_mtde_net_metrics, split_by_sequence

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Carga del Dataset y Partición Estricta por Secuencias

In [3]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
time_scale = 30.0

train_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata_train.csv", 
    is_train=True, 
    min_time_s=0.0,
    time_scale=time_scale
)
val_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata_train.csv", 
    is_train=False, 
    min_time_s=0.0,
    time_scale=time_scale
)
train_ds_full.root = Path("../processed_data")
val_ds_full.root = Path("../processed_data")

t_idx, v_idx = split_by_sequence(train_ds_full.df)
print(f"Dataset cargado exitosamente. Particiones: {len(t_idx)} train / {len(v_idx)} val")

Dataset cargado exitosamente. Particiones: 1111 train / 329 val


### 3. Función Genérica de Entrenamiento y Evaluación

In [4]:
def train_and_evaluate(backbone_name, pretrained, freeze_backbone, lr, epochs=30, batch_size=16, patience=8):
    set_seed(42)
    
    train_loader = DataLoader(Subset(train_ds_full, t_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(val_ds_full, v_idx), batch_size=batch_size)
    
    # Instanciar el modelo con transfer learning
    model = MTDE_Net_TL(
        backbone_name=backbone_name,
        pretrained=pretrained,
        freeze_backbone=freeze_backbone,
        tabular_dim=4,
        dropout=0.2
    ).to(dev)
    
    crit = SqrtScaledMSELoss(scale=time_scale)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    best_mae = float("inf")
    best_metrics = None
    patience_counter = 0
    
    print(f"\n>>> Entrenando {backbone_name.upper()} | Pretrained: {pretrained} | Freeze: {freeze_backbone} | lr: {lr}")
    
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for x_img, x_tab, y in train_loader:
            x_img, x_tab, y = x_img.to(dev), x_tab.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x_img, x_tab), y)
            loss.backward()
            opt.step()
            train_loss += loss.item() * len(y)
            
        train_loss /= len(t_idx)
        
        # Evaluar en validación
        metrics = eval_mtde_net_metrics(model, val_loader, dev, scale=time_scale)
        v_mae = metrics["mae"]
        
        # Impresión del progreso
        star = ""
        if v_mae < best_mae:
            best_mae = v_mae
            best_metrics = metrics
            patience_counter = 0
            star = " *"
            torch.save(model.state_dict(), f"../MTDE_Net_TL_{backbone_name}_{'congelado' if freeze_backbone else 'finetuned'}.pt")
        else:
            patience_counter += 1
            
        print(f"Ep {ep:03d} | Loss: {train_loss:.4f} | ValMAE: {v_mae:.2f}s | RMSE: {metrics['rmse']:.2f}s | R2: {metrics['r2']:.4f} | MAPE: {metrics['mape']:.2f}% | Acc60: {metrics['acc60']:.2f}% | Acc120: {metrics['acc120']:.2f}%{star}")
        
        if patience_counter >= patience:
            print(f"Early stop alcanzado. Mejor Val MAE: {best_mae:.2f}s")
            break
            
    return best_metrics

### 4. Experimento 1: ResNet-18 Preentrenado + Fine-tuning completo

In [5]:
# Fine-tuning completo con una lr baja de 2e-4
m_r18_ft = train_and_evaluate(
    backbone_name="resnet18", 
    pretrained=True, 
    freeze_backbone=False, 
    lr=2e-4, 
    epochs=35, 
    batch_size=16,
    patience=10
)


>>> Entrenando RESNET18 | Pretrained: True | Freeze: False | lr: 0.0002
Ep 001 | Loss: 0.0484 | ValMAE: 104.77s | RMSE: 134.95s | R2: 0.3300 | MAPE: 64.55% | Acc60: 32.52% | Acc120: 72.64% *
Ep 002 | Loss: 0.0277 | ValMAE: 85.15s | RMSE: 110.79s | R2: 0.5484 | MAPE: 56.18% | Acc60: 49.54% | Acc120: 77.20% *
Ep 003 | Loss: 0.0191 | ValMAE: 76.38s | RMSE: 95.04s | R2: 0.6677 | MAPE: 54.39% | Acc60: 47.11% | Acc120: 83.59% *
Ep 004 | Loss: 0.0139 | ValMAE: 67.12s | RMSE: 89.17s | R2: 0.7075 | MAPE: 48.77% | Acc60: 58.97% | Acc120: 85.11% *
Ep 005 | Loss: 0.0102 | ValMAE: 53.08s | RMSE: 74.65s | R2: 0.7950 | MAPE: 38.38% | Acc60: 65.35% | Acc120: 90.27% *
Ep 006 | Loss: 0.0079 | ValMAE: 43.89s | RMSE: 58.30s | R2: 0.8750 | MAPE: 34.47% | Acc60: 73.25% | Acc120: 95.74% *
Ep 007 | Loss: 0.0079 | ValMAE: 46.56s | RMSE: 65.49s | R2: 0.8422 | MAPE: 34.02% | Acc60: 75.68% | Acc120: 91.49%
Ep 008 | Loss: 0.0062 | ValMAE: 29.76s | RMSE: 42.70s | R2: 0.9329 | MAPE: 27.06% | Acc60: 89.97% | Acc120:

### 5. Experimento 2: ResNet-18 Congelado (Feature Extractor estático)

In [6]:
# Backbone congelado, se puede usar una lr un poco mayor (5e-4) para el cabezal
m_r18_frozen = train_and_evaluate(
    backbone_name="resnet18", 
    pretrained=True, 
    freeze_backbone=True, 
    lr=5e-4, 
    epochs=35, 
    batch_size=16,
    patience=10
)


>>> Entrenando RESNET18 | Pretrained: True | Freeze: True | lr: 0.0005
Ep 001 | Loss: 0.0569 | ValMAE: 114.74s | RMSE: 165.83s | R2: -0.0117 | MAPE: 64.74% | Acc60: 48.02% | Acc120: 67.78% *
Ep 002 | Loss: 0.0238 | ValMAE: 88.87s | RMSE: 133.11s | R2: 0.3481 | MAPE: 55.07% | Acc60: 55.02% | Acc120: 77.81% *
Ep 003 | Loss: 0.0132 | ValMAE: 73.76s | RMSE: 114.33s | R2: 0.5191 | MAPE: 86.81% | Acc60: 61.40% | Acc120: 82.07% *
Ep 004 | Loss: 0.0107 | ValMAE: 61.87s | RMSE: 92.26s | R2: 0.6869 | MAPE: 102.76% | Acc60: 67.48% | Acc120: 84.19% *
Ep 005 | Loss: 0.0097 | ValMAE: 61.96s | RMSE: 93.02s | R2: 0.6817 | MAPE: 79.84% | Acc60: 65.05% | Acc120: 83.28%
Ep 006 | Loss: 0.0093 | ValMAE: 58.26s | RMSE: 87.07s | R2: 0.7211 | MAPE: 86.96% | Acc60: 66.57% | Acc120: 85.41% *
Ep 007 | Loss: 0.0091 | ValMAE: 63.82s | RMSE: 96.44s | R2: 0.6578 | MAPE: 78.32% | Acc60: 64.13% | Acc120: 83.28%
Ep 008 | Loss: 0.0091 | ValMAE: 59.22s | RMSE: 89.51s | R2: 0.7052 | MAPE: 81.44% | Acc60: 68.09% | Acc120:

### 6. Experimento 3: ResNet-50 Preentrenado + Fine-tuning completo

In [7]:
# ResNet-50 es más expresivo y denso. Usamos una lr controlada de 1e-4
m_r50_ft = train_and_evaluate(
    backbone_name="resnet50", 
    pretrained=True, 
    freeze_backbone=False, 
    lr=1e-4, 
    epochs=35, 
    batch_size=16,
    patience=10
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\esteb/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:04<00:00, 24.0MB/s]



>>> Entrenando RESNET50 | Pretrained: True | Freeze: False | lr: 0.0001
Ep 001 | Loss: 0.0635 | ValMAE: 132.84s | RMSE: 181.90s | R2: -0.2173 | MAPE: 70.17% | Acc60: 37.08% | Acc120: 62.31% *
Ep 002 | Loss: 0.0411 | ValMAE: 132.16s | RMSE: 169.67s | R2: -0.0590 | MAPE: 75.32% | Acc60: 26.75% | Acc120: 57.45% *
Ep 003 | Loss: 0.0331 | ValMAE: 112.77s | RMSE: 146.33s | R2: 0.2123 | MAPE: 67.76% | Acc60: 30.70% | Acc120: 69.30% *
Ep 004 | Loss: 0.0269 | ValMAE: 98.10s | RMSE: 127.70s | R2: 0.4001 | MAPE: 60.37% | Acc60: 40.12% | Acc120: 74.16% *
Ep 005 | Loss: 0.0234 | ValMAE: 90.88s | RMSE: 117.37s | R2: 0.4932 | MAPE: 58.77% | Acc60: 41.03% | Acc120: 77.20% *
Ep 006 | Loss: 0.0192 | ValMAE: 79.47s | RMSE: 106.06s | R2: 0.5862 | MAPE: 53.35% | Acc60: 56.53% | Acc120: 79.94% *
Ep 007 | Loss: 0.0160 | ValMAE: 71.20s | RMSE: 94.31s | R2: 0.6728 | MAPE: 49.13% | Acc60: 59.88% | Acc120: 84.19% *
Ep 008 | Loss: 0.0135 | ValMAE: 70.90s | RMSE: 92.01s | R2: 0.6885 | MAPE: 51.71% | Acc60: 56.84%

### 7. Resumen y Comparación de Resultados Experimentales

In [8]:
results_data = []

if m_r18_ft:
    results_data.append({
        "Configuración": "ResNet-18 Fine-tuned (ImageNet)",
        "MAE (s)": f"{m_r18_ft['mae']:.2f} s",
        "RMSE (s)": f"{m_r18_ft['rmse']:.2f} s",
        "R²": f"{m_r18_ft['r2']:.4f}",
        "MAPE (%)": f"{m_r18_ft['mape']:.2f} %",
        "Acc@60s (%)": f"{m_r18_ft['acc60']:.2f} %",
        "Acc@120s (%)": f"{m_r18_ft['acc120']:.2f} %"
    })
if m_r18_frozen:
    results_data.append({
        "Configuración": "ResNet-18 Frozen (FE)",
        "MAE (s)": f"{m_r18_frozen['mae']:.2f} s",
        "RMSE (s)": f"{m_r18_frozen['rmse']:.2f} s",
        "R²": f"{m_r18_frozen['r2']:.4f}",
        "MAPE (%)": f"{m_r18_frozen['mape']:.2f} %",
        "Acc@60s (%)": f"{m_r18_frozen['acc60']:.2f} %",
        "Acc@120s (%)": f"{m_r18_frozen['acc120']:.2f} %"
    })
if m_r50_ft:
    results_data.append({
        "Configuración": "ResNet-50 Fine-tuned (ImageNet)",
        "MAE (s)": f"{m_r50_ft['mae']:.2f} s",
        "RMSE (s)": f"{m_r50_ft['rmse']:.2f} s",
        "R²": f"{m_r50_ft['r2']:.4f}",
        "MAPE (%)": f"{m_r50_ft['mape']:.2f} %",
        "Acc@60s (%)": f"{m_r50_ft['acc60']:.2f} %",
        "Acc@120s (%)": f"{m_r50_ft['acc120']:.2f} %"
    })

df_res = pd.DataFrame(results_data)
print("\n=== RESUMEN COMPARATIVO DE TRANSFER LEARNING ===")
display(df_res)


=== RESUMEN COMPARATIVO DE TRANSFER LEARNING ===


,Configuración,MAE (s),RMSE (s),R²,MAPE (%),Acc@60s (%),Acc@120s (%)
0,ResNet-18 Fine-tuned (ImageNet),25.70 s,35.06 s,0.9548,27.42 %,92.10 %,98.78 %
1,ResNet-18 Frozen (FE),53.09 s,80.81 s,0.7597,60.86 %,69.91 %,86.63 %
2,ResNet-50 Fine-tuned (ImageNet),25.83 s,40.78 s,0.9388,31.78 %,86.93 %,98.48 %
